## Implementation of concepts

Paper: *Distilling the Knowledge in a Neural Network*

#### Concepts:
1. Distilled model of the same size, from a cumbersome model that generalizes well, will generalize better than models of the same size without any distillation
2. Higher temperature in the softmax appears to transfer better when student model has much less parameter counts
3. 

### Section: Training a cumbersome model

The paper described their cumbersome "master" model as the a MLP with two hidden layers with 1200 ReLU units each and then a softmax in the end. 

They decsribed that for the cumbersome model they were able to achieve a test error of 0.67%, we will try to replicate this result and aim for 0.7% test results.

In [1]:
# Downloading/loading the MNIST dataset
from llm_distill.dataset.mnist import load_mnist

x_train, y_train = load_mnist("train")
x_test, y_test = load_mnist("test")

In [2]:
import torch
from torch import nn


class CumbersomeNet(nn.Module):
    """Teacher MLP from Sec. 3 of Hinton et al. 2015: 784 -> 1200 -> 1200 -> 10 with ReLU and dropout."""

    def __init__(self, hidden: int = 1200, p_input: float = 0.2, p_hidden: float = 0.5) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p_input),
            nn.Linear(28 * 28, hidden),
            nn.ReLU(),
            nn.Dropout(p_hidden),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(p_hidden),
            nn.Linear(hidden, 10),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Map images (N, 1, 28, 28) or (N, 784) to logits (N, 10)."""
        return self.net(x)


teacher = CumbersomeNet()
print(teacher)
print(f"parameters: {sum(p.numel() for p in teacher.parameters()):,}")

CumbersomeNet(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.2, inplace=False)
    (2): Linear(in_features=784, out_features=1200, bias=True)
    (3): ReLU()
    (4): Dropout(p=0.5, inplace=False)
    (5): Linear(in_features=1200, out_features=1200, bias=True)
    (6): ReLU()
    (7): Dropout(p=0.5, inplace=False)
    (8): Linear(in_features=1200, out_features=10, bias=True)
  )
)
parameters: 2,395,210


In [3]:
# Importing training function:
from llm_distill.utils.train import train, TrainConfig
from torch.utils.data import TensorDataset

train_config = TrainConfig(
    epochs=50,
    batch_size=512,
    optimizer="adamw",
    lr=1e-3,
    weight_decay=1e-2,
    scheduler="cosine",
    eval_strategy="epoch",
    log_every_n_batches=10,
    verbose=False
)

x_train_scaled = x_train.float() / 255.0
mean, std = x_train_scaled.mean().item(), x_train_scaled.std().item()

def standardize(x: torch.Tensor, mean, std) -> torch.Tensor:
    return (x.float() / 255.0 - mean) / std


train_ds = TensorDataset(standardize(x_train, mean, std), y_train)
test_ds = TensorDataset(standardize(x_test, mean, std), y_test)

naive_training_history = train(teacher, train_ds, test_ds, train_config)


In [4]:
for e, ev in zip(naive_training_history.epochs, naive_training_history.eval, strict=True):
    print(
        f"epoch {e['epoch']:2d}  train loss {e['loss']:.4f}  acc {e['accuracy']:.4f}  |  "
        f"test loss {ev['loss']:.4f}  acc {ev['accuracy']:.4f}  errors {ev['errors']}"
    )

epoch  1  train loss 0.3768  acc 0.8833  |  test loss 0.1338  acc 0.9600  errors 400
epoch  2  train loss 0.1679  acc 0.9477  |  test loss 0.0927  acc 0.9715  errors 285
epoch  3  train loss 0.1297  acc 0.9595  |  test loss 0.0816  acc 0.9746  errors 254
epoch  4  train loss 0.1143  acc 0.9643  |  test loss 0.0772  acc 0.9746  errors 254
epoch  5  train loss 0.0989  acc 0.9684  |  test loss 0.0704  acc 0.9772  errors 228
epoch  6  train loss 0.0918  acc 0.9717  |  test loss 0.0608  acc 0.9812  errors 188
epoch  7  train loss 0.0842  acc 0.9730  |  test loss 0.0601  acc 0.9799  errors 201
epoch  8  train loss 0.0779  acc 0.9751  |  test loss 0.0545  acc 0.9825  errors 175
epoch  9  train loss 0.0715  acc 0.9764  |  test loss 0.0561  acc 0.9826  errors 174
epoch 10  train loss 0.0681  acc 0.9777  |  test loss 0.0549  acc 0.9833  errors 167
epoch 11  train loss 0.0620  acc 0.9797  |  test loss 0.0546  acc 0.9842  errors 158
epoch 12  train loss 0.0607  acc 0.9802  |  test loss 0.0529  acc

After training with a variety of different hyperparameters for the cumbersome model, the test error hovers above 1 to 1.5%. With agressive dropout already implemented, the only missing pieces are the input jitters (paper said they shifted each training image by up to 2 pixels):

In [5]:
import torch.nn.functional as F
from torch.utils.data import Dataset

class JitterDataset(Dataset):
    """Randomly shifts and horizontally flips MNIST images."""

    def __init__(
        self,
        x: torch.Tensor,
        y: torch.Tensor,
        max_shift: int = 2,
        flip_prob: float = 0.1,
        fill: float = 0.0,
    ) -> None:
        self.s = max_shift
        self.flip_prob = flip_prob
        self.x = F.pad(x, (max_shift,) * 4, value=fill)
        self.y = y

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        i, j = torch.randint(0, 2 * self.s + 1, (2,)).tolist()
        image = self.x[idx, i : i + 28, j : j + 28]

        if torch.rand(()) .item() < self.flip_prob:
            image = torch.flip(image, dims=(-1,))

        return image, self.y[idx]


background = (0.0 - mean) / std  # standardized value of a black pixel

In [6]:
train_config = TrainConfig(
    epochs=100,
    batch_size=512,
    optimizer="adamw",
    lr=1e-3,
    weight_decay=1e-2,
    scheduler="cosine",
    eval_strategy="epoch",
    log_every_n_batches=10,
    verbose=False
)

In [7]:
teacher_shift2 = CumbersomeNet()
train_ds_shift2 = JitterDataset(standardize(x_train, mean, std), y_train, max_shift=2, flip_prob=0.02, fill=background)
training_history_shift2 = train(teacher_shift2, train_ds_shift2, test_ds, train_config)

teacher_shift2_flip2 = CumbersomeNet()
train_ds_shift2_flip2 = JitterDataset(standardize(x_train, mean, std), y_train, max_shift=2, flip_prob=0.02, fill=background)
training_history_shift2_flip2 = train(teacher_shift2_flip2, train_ds_shift2_flip2, test_ds, train_config)


teacher_shift5 = CumbersomeNet()
train_ds_shift5 = JitterDataset(standardize(x_train, mean, std), y_train, max_shift=5, flip_prob=0.02, fill=background)
training_history_shift5 = train(teacher_shift5, train_ds_shift5, test_ds, train_config)

In [8]:
import plotly.graph_objects as go

train_logs = {
    "no_jitter": naive_training_history,
    "shift2": training_history_shift2,
    "shift2_flip2": training_history_shift2_flip2,
    "shift5": training_history_shift5,
}

eval_length = max([len(history.eval) for history in train_logs.values()])

fig = go.Figure()

for name, history in train_logs.items():
    fig.add_trace(
        go.Scatter(
            x=list(range(eval_length)), y=[i["accuracy"] for i in history.eval], mode='lines', name=name
        )
    )

# add dotted line at 99.33% accuracy (0.67% error)
fig.add_trace(
    go.Scatter(
        x=[0, eval_length - 1],
        y=[0.9933, 0.9933],
        mode='lines',
        name='target accuracy',
        line=dict(dash='dot', color='black')
    )
)

fig.update_layout(
    title="Test Accuracy of Cumbersome Models with Input Jitters",
    xaxis_title="Epochs",
    yaxis_title="Test Accuracy"
)



fig.show()

With the jitter with max 2 pixel shift, the test error is now around 0.8%, with some best model trains achieving 0.7% test error. We will train again but this time saving the best model based on the test error, and we will use that for further distillation experiments.

Will save it to `experiments/hinton2015_mnist/teacher.pt`.

In [12]:
final_teacher = CumbersomeNet()
train_ds = JitterDataset(
    standardize(x_train, mean, std), y_train, max_shift=2, fill=background
)

# Train config:

train_config = TrainConfig(
    epochs=100,
    batch_size=512,
    optimizer="adamw",
    lr=1e-3,
    weight_decay=1e-2,
    scheduler="cosine",
    eval_strategy="epoch",
    log_every_n_batches=10,
    verbose=False
)

train_history = train(
    final_teacher, train_ds, test_ds, train_config,
    save_best_model="experiments/hinton2015_mnist/teacher.pt",
    best_metric="test_acc"
)